# Project 53 — Local SQL Analyst Agent
## Natural Language → SQL → Results → Summary

**Stack:** LangChain · Ollama · SQLite · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community

## Step 1 — Create Sample SQLite Database

In [ ]:
import sqlite3
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)
conn = sqlite3.connect("sample_data/company.db")
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS employees;
DROP TABLE IF EXISTS departments;
DROP TABLE IF EXISTS projects;

CREATE TABLE departments (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    budget REAL
);

CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    dept_id INTEGER REFERENCES departments(id),
    salary REAL,
    hire_date TEXT,
    role TEXT
);

CREATE TABLE projects (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    dept_id INTEGER REFERENCES departments(id),
    status TEXT,
    budget_used REAL
);

INSERT INTO departments VALUES (1,'Engineering',500000),(2,'Marketing',200000),(3,'Sales',300000);

INSERT INTO employees VALUES
(1,'Alice',1,120000,'2022-01-15','Senior Engineer'),
(2,'Bob',1,95000,'2023-03-01','Engineer'),
(3,'Carol',2,85000,'2022-06-15','Marketing Manager'),
(4,'Dave',3,90000,'2021-11-01','Sales Lead'),
(5,'Eve',1,110000,'2022-09-01','Staff Engineer'),
(6,'Frank',2,75000,'2024-01-10','Marketing Associate'),
(7,'Grace',3,82000,'2023-07-01','Sales Rep'),
(8,'Hank',1,130000,'2021-03-15','Engineering Manager');

INSERT INTO projects VALUES
(1,'API Redesign',1,'active',150000),
(2,'Brand Refresh',2,'completed',80000),
(3,'Q1 Campaign',3,'active',120000),
(4,'ML Pipeline',1,'active',200000),
(5,'Website Relaunch',2,'planning',50000);
""")
conn.commit()

# Show schema
for table in ['departments', 'employees', 'projects']:
    cur.execute(f"SELECT * FROM {table}")
    rows = cur.fetchall()
    print(f"\n{table}: {len(rows)} rows")
    for row in rows[:3]:
        print(f"  {row}")
conn.close()
print("\nDatabase created!")

## Step 2 — Build NL-to-SQL Agent

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

schema = """
departments(id, name, budget)
employees(id, name, dept_id, salary, hire_date, role)
projects(id, name, dept_id, status, budget_used)
"""

def nl_to_sql(question):
    # Step 1: Generate SQL
    sql_prompt = ChatPromptTemplate.from_messages([
        ("system", f"""You are a SQL expert. Given this schema:
{schema}
Generate a SQLite query to answer the question. Return ONLY the SQL, no explanation."""),
        ("human", "{question}")
    ])
    sql_chain = sql_prompt | llm | StrOutputParser()
    sql = sql_chain.invoke({"question": question}).strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()

    # Step 2: Execute
    conn = sqlite3.connect("sample_data/company.db")
    try:
        results = conn.execute(sql).fetchall()
        columns = [desc[0] for desc in conn.execute(sql).description] if results else []
    except Exception as e:
        conn.close()
        return {"sql": sql, "error": str(e)}
    conn.close()

    # Step 3: Summarize
    summary_prompt = ChatPromptTemplate.from_messages([
        ("system", "Summarize the SQL results in natural language."),
        ("human", "Question: {question}\nSQL: {sql}\nResults: {results}")
    ])
    summary_chain = summary_prompt | llm | StrOutputParser()
    summary = summary_chain.invoke({
        "question": question, "sql": sql,
        "results": str(list(zip(columns, *zip(*results))) if results else "No results"),
    })

    return {"sql": sql, "results": results, "columns": columns, "summary": summary}

print("SQL analyst ready!")

## Step 3 — Run Queries

In [ ]:
questions = [
    "What is the average salary by department?",
    "List all active projects and their budgets",
    "Who is the highest paid employee?",
    "How many employees were hired in 2023?",
    "What percentage of the engineering budget has been used by projects?",
]

for q in questions:
    print(f"\nQ: {q}")
    result = nl_to_sql(q)
    if "error" in result:
        print(f"  SQL: {result['sql']}")
        print(f"  Error: {result['error']}")
    else:
        print(f"  SQL: {result['sql']}")
        print(f"  Results: {result['results'][:5]}")
        print(f"  Summary: {result['summary']}")

## What You Learned
- **NL-to-SQL generation** with schema context
- **Safe SQL execution** with error handling
- **Result summarization** in natural language